# L4: 数据模型与验证

> 使用 Pydantic 进行数据验证和序列化

In [1]:
# === 环境设置 ===
import sys
import os
import json
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")

# 验证模块导入
try:
    from pydantic import BaseModel, Field, field_validator
    print("✓ Pydantic 导入成功")
except ImportError as e:
    print(f"✗ Pydantic 导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
✓ Pydantic 导入成功


## 4.1 什么是 Pydantic？

**Pydantic** 是 Python 的数据验证库，使用类型注解进行数据验证。

### 为什么需要数据验证？

```
没有验证:
  user_input = {"age": "25"}  # 字符串！
  age + 1 = "251"  # 字符串拼接，错误！

有验证:
  user.age = int(25)  # 自动转换
  age + 1 = 26  # 正确！
```

## 4.2 Pydantic 基础

### 定义一个简单的模型

In [2]:
from pydantic import BaseModel, Field
from typing import Optional

# 定义一个用户模型
class User(BaseModel):
    name: str
    age: int
    email: Optional[str] = None

# 创建实例
user1 = User(name="Alice", age=25, email="alice@example.com")
print(f"用户: {user1.name}, 年龄: {user1.age}, 邮箱: {user1.email}")

# 自动类型转换
user2 = User(name="Bob", age="30", email=None)  # age 是字符串，但会自动转换
print(f"\n自动转换:")
print(f"  age 类型: {type(user2.age)}")
print(f"  age 值: {user2.age}")

用户: Alice, 年龄: 25, 邮箱: alice@example.com

自动转换:
  age 类型: <class 'int'>
  age 值: 30


### 数据验证演示

In [3]:
from pydantic import ValidationError, field_validator

class StrictUser(BaseModel):
    name: str
    age: int
    
    @field_validator('age')
    @classmethod
    def check_age(cls, v):
        if v < 0:
            raise ValueError('age must be positive')
        if v > 150:
            raise ValueError('age must be realistic')
        return v

# 正常情况
try:
    user = StrictUser(name="Charlie", age=30)
    print(f"✓ 验证通过: {user.name}, {user.age}岁")
except ValidationError as e:
    print(f"✗ 验证失败: {e}")

# 异常情况
print("\n测试异常输入:")
test_cases = [
    {"name": "Dave", "age": -5},
    {"name": "Eve", "age": 200},
]

for data in test_cases:
    try:
        user = StrictUser(**data)
        print(f"  {data} → ✓")
    except ValidationError as e:
        print(f"  {data} → ✗ {e.errors()[0]['msg']}")

✓ 验证通过: Charlie, 30岁

测试异常输入:
  {'name': 'Dave', 'age': -5} → ✗ Value error, age must be positive
  {'name': 'Eve', 'age': 200} → ✗ Value error, age must be realistic


## 4.3 Field 的使用

### Field 用于添加元数据和约束

In [4]:
from pydantic import BaseModel, Field
from typing import List

class Message(BaseModel):
    role: str = Field(..., description="消息角色: system/user/assistant/tool")
    content: str = Field(..., min_length=1, max_length=10000, description="消息内容")
    metadata: dict = Field(default_factory=dict, description="附加元数据")
    score: float = Field(ge=0, le=1, description="置信度分数 0-1")

# 创建消息
msg = Message(
    role="user",
    content="你好，世界！",
    score=0.95
)

print(f"消息: {msg.role}: {msg.content}")
print(f"置信度: {msg.score}")

# 查看 JSON Schema
print("\nJSON Schema:")
print(json.dumps(Message.model_json_schema(), indent=2, ensure_ascii=False))

消息: user: 你好，世界！
置信度: 0.95

JSON Schema:
{
  "properties": {
    "role": {
      "description": "消息角色: system/user/assistant/tool",
      "title": "Role",
      "type": "string"
    },
    "content": {
      "description": "消息内容",
      "maxLength": 10000,
      "minLength": 1,
      "title": "Content",
      "type": "string"
    },
    "metadata": {
      "additionalProperties": true,
      "description": "附加元数据",
      "title": "Metadata",
      "type": "object"
    },
    "score": {
      "description": "置信度分数 0-1",
      "maximum": 1,
      "minimum": 0,
      "title": "Score",
      "type": "number"
    }
  },
  "required": [
    "role",
    "content",
    "score"
  ],
  "title": "Message",
  "type": "object"
}


### Field 常用参数

| 参数 | 作用 | 示例 |
|------|------|------|
| `...` | 必填字段 | `name: str = Field(...)` |
| `default` | 默认值 | `age: int = Field(default=0)` |
| `default_factory` | 默认工厂函数 | `items: list = Field(default_factory=list)` |
| `description` | 字段描述 | `name: str = Field(description="用户名")` |
| `ge/le/gt/lt` | 数值范围 | `age: int = Field(ge=0, le=150)` |
| `min_length/max_length` | 字符串长度 | `name: str = Field(min_length=2)` |
| `pattern` | 正则匹配 | `phone: str = Field(pattern=r'^\d+$')` |

## 4.4 嵌套模型

### 模型可以包含其他模型

In [5]:
from pydantic import BaseModel
from typing import List, Optional

class Address(BaseModel):
    street: str
    city: str
    country: str = "China"

class Person(BaseModel):
    name: str
    age: int
    address: Address  # 嵌套模型
    friends: Optional[List[str]] = None

# 创建带嵌套的数据
person = Person(
    name="张三",
    age=25,
    address={
        "street": "中关村大街1号",
        "city": "北京",
        "country": "中国"
    },
    friends=["李四", "王五"]
)

print(f"姓名: {person.name}")
print(f"地址: {person.address.city}, {person.address.street}")
print(f"朋友: {person.friends}")

# 转换为 JSON
print("\nJSON 输出:")
print(person.model_dump_json(indent=2, ensure_ascii=False))

姓名: 张三
地址: 北京, 中关村大街1号
朋友: ['李四', '王五']

JSON 输出:
{
  "name": "张三",
  "age": 25,
  "address": {
    "street": "中关村大街1号",
    "city": "北京",
    "country": "中国"
  },
  "friends": [
    "李四",
    "王五"
  ]
}


## 4.5 项目中的数据模型

### 查看 LLM 相关的模型

In [6]:
from backend.llm.models import Message, ChatResponse, TokenUsage, ToolCall     
print("=== 项目中的数据模型 ===\n")                                         

# Message 模型                                                              
print("Message 模型的字段:")
for field_name, field_info in Message.model_fields.items():
    required = "必填" if field_info.is_required() else "可选"
    desc = field_info.description or "无描述"
    print(f"  - {field_name}: {str(field_info.annotation)} ({required}) - {desc}")

print("\n" + "="*50)

# TokenUsage 模型
print("\nTokenUsage 模型的字段:")
for field_name, field_info in TokenUsage.model_fields.items():
    print(f"  - {field_name}: {str(field_info.annotation)} - {field_info.description}")

=== 项目中的数据模型 ===

Message 模型的字段:
  - role: <class 'str'> (必填) - 消息角色: system/user/assistant/tool
  - content: <class 'str'> (必填) - 消息内容
  - tool_call_id: str | None (可选) - 工具调用ID（tool消息用）
  - tool_calls: typing.List[backend.llm.models.ToolCall] | None (可选) - 工具调用列表（assistant消息用）
  - metadata: typing.Dict[str, typing.Any] | None (可选) - 附加元数据


TokenUsage 模型的字段:
  - prompt_tokens: <class 'int'> - 输入token数
  - completion_tokens: <class 'int'> - 输出token数
  - total_tokens: <class 'int'> - 总token数


### 创建和操作 Message 对象

In [7]:
# 创建消息
system_msg = Message(
    role="system",
    content="你是一个友好的AI助手。",
    metadata={"temperature": 0.7}
)

user_msg = Message(
    role="user",
    content="什么是Python？"
)

print(f"系统消息: {system_msg.role}: {system_msg.content}")
print(f"用户消息: {user_msg.role}: {user_msg.content}")

# 转换为字典
print("\n转换为字典:")
print(system_msg.model_dump())

# 转换为 JSON
print("\n转换为 JSON:")
print(system_msg.model_dump_json())

系统消息: system: 你是一个友好的AI助手。
用户消息: user: 什么是Python？

转换为字典:
{'role': 'system', 'content': '你是一个友好的AI助手。', 'tool_call_id': None, 'tool_calls': None, 'metadata': {'temperature': 0.7}}

转换为 JSON:
{"role":"system","content":"你是一个友好的AI助手。","tool_call_id":null,"tool_calls":null,"metadata":{"temperature":0.7}}


## 4.6 JSON Schema 生成

### Pydantic 可以自动生成 JSON Schema

In [8]:
# 为 Message 模型生成 JSON Schema
schema = Message.model_json_schema()
print(schema)
print("Message 的 JSON Schema:")
print(json.dumps(schema, indent=2, ensure_ascii=False))

{'$defs': {'FunctionCall': {'description': '函数调用信息', 'properties': {'name': {'description': '函数名称', 'title': 'Name', 'type': 'string'}, 'arguments': {'description': '函数参数（JSON字符串）', 'title': 'Arguments', 'type': 'string'}}, 'required': ['name', 'arguments'], 'title': 'FunctionCall', 'type': 'object'}, 'ToolCall': {'description': '工具调用模型', 'properties': {'id': {'description': '工具调用ID', 'title': 'Id', 'type': 'string'}, 'type': {'default': 'function', 'description': '调用类型', 'title': 'Type', 'type': 'string'}, 'function': {'$ref': '#/$defs/FunctionCall'}}, 'required': ['id', 'function'], 'title': 'ToolCall', 'type': 'object'}}, 'description': '聊天消息模型', 'properties': {'role': {'description': '消息角色: system/user/assistant/tool', 'title': 'Role', 'type': 'string'}, 'content': {'description': '消息内容', 'title': 'Content', 'type': 'string'}, 'tool_call_id': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': '工具调用ID（tool消息用）', 'title': 'Tool Call Id'}, 'tool_calls': 

### 为什么需要 JSON Schema？

```
1. Function Calling: 告诉 LLM 工具的参数格式
2. API 文档: 自动生成接口文档
3. 前后端对接: 确保数据格式一致
4. 数据存储: NoSQL 数据库的 schema 验证
```

## 4.7 实战练习：定义聊天模型

### 练习 1: 定义一个简单的聊天请求模型

In [9]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional, Literal

class ChatMessage(BaseModel):
    """聊天消息模型"""
    role: Literal["system", "user", "assistant"] = Field(
        ..., 
        description="消息角色"
    )
    content: str = Field(
        ..., 
        min_length=1, 
        max_length=5000,
        description="消息内容"
    )
    
class ChatRequest(BaseModel):
    """聊天请求模型"""
    messages: List[ChatMessage] = Field(
        ..., 
        min_length=1,
        description="消息列表"
    )
    temperature: float = Field(
        default=0.7,
        ge=0, 
        le=2,
        description="温度参数 0-2"
    )
    max_tokens: Optional[int] = Field(
        None,
        gt=0,
        description="最大生成 token 数"
    )
    stream: bool = Field(
        default=False,
        description="是否流式输出"
    )
    
# 测试模型
request = ChatRequest(
    messages=[
        ChatMessage(role="system", content="你是一个助手"),
        ChatMessage(role="user", content="你好"),
    ],
    temperature=0.5,
    stream=False
)

print("聊天请求:")
print(f"  消息数量: {len(request.messages)}")
print(f"  温度: {request.temperature}")
print(f"  流式: {request.stream}")

# 生成 JSON Schema（用于 Function Calling）
print("\nJSON Schema:")
schema = ChatRequest.model_json_schema()
print(json.dumps(schema['properties'], indent=2, ensure_ascii=False))

聊天请求:
  消息数量: 2
  温度: 0.5
  流式: False

JSON Schema:
{
  "messages": {
    "description": "消息列表",
    "items": {
      "$ref": "#/$defs/ChatMessage"
    },
    "minItems": 1,
    "title": "Messages",
    "type": "array"
  },
  "temperature": {
    "default": 0.7,
    "description": "温度参数 0-2",
    "maximum": 2,
    "minimum": 0,
    "title": "Temperature",
    "type": "number"
  },
  "max_tokens": {
    "anyOf": [
      {
        "exclusiveMinimum": 0,
        "type": "integer"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "最大生成 token 数",
    "title": "Max Tokens"
  },
  "stream": {
    "default": false,
    "description": "是否流式输出",
    "title": "Stream",
    "type": "boolean"
  }
}


### 练习 2: 定义工具调用模型

In [10]:
from pydantic import BaseModel, Field

class FunctionCall(BaseModel):
    """函数调用"""
    name: str = Field(..., description="函数名称")
    arguments: str = Field(..., description="参数 JSON 字符串")

class ToolCall(BaseModel):
    """工具调用"""
    id: str = Field(..., description="调用ID")
    type: str = Field(default="function", description="调用类型")
    function: FunctionCall = Field(..., description="函数信息")

# 创建工具调用
tool_call = ToolCall(
    id="call_123",
    function=FunctionCall(
        name="calculator",
        arguments='{"expression": "2 + 2"}'
    )
)

print("工具调用:")
print(f"  ID: {tool_call.id}")
print(f"  函数: {tool_call.function.name}")
print(f"  参数: {tool_call.function.arguments}")

# 转换为字典（用于 API 调用）
print("\n转换为字典:")
print(tool_call.model_dump())

工具调用:
  ID: call_123
  函数: calculator
  参数: {"expression": "2 + 2"}

转换为字典:
{'id': 'call_123', 'type': 'function', 'function': {'name': 'calculator', 'arguments': '{"expression": "2 + 2"}'}}


## 4.8 常见面试问题

**Q: Pydantic 和 dataclass 有什么区别？**
- A: 
  | | Pydantic | dataclass |
  |-|----------|----------|
  | **类型转换** | ✅ 自动转换 | ❌ 不转换 |
  | **数据验证** | ✅ 强大的验证 | ❌ 无验证 |
  | **JSON Schema** | ✅ 自动生成 | ❌ 需手动实现 |
  | **性能** | 较慢（有验证开销）| 更快 |
  | **依赖** | 需安装 pydantic | 内置 |

**Q: Field(...) 中的 ... 是什么？**
- A: `...` (Ellipsis) 表示该字段是必填的，没有默认值。
  ```python
  name: str = Field(...)     # 必填
  age: int = Field(default=0)  # 可选，默认0
  ```

**Q: 如何处理可选字段？**
- A: 使用 `Optional` 类型或提供默认值：
  ```python
  from typing import Optional
  
  # 方式1: Optional
  email: Optional[str] = None
  
  # 方式2: 直接提供默认
  email: str = None  # Pydantic v2 支持
  
  # 方式3: Field with default
  email: Optional[str] = Field(default=None)
  ```

**Q: 什么是 JSON Schema？为什么需要它？**
- A: JSON Schema 是一种描述 JSON 数据结构的格式。
  - **用途**: 前后端对接、API 文档、数据验证、Function Calling
  - **示例**: 告诉 LLM 某个工具需要什么参数

**Q: Pydantic 的 model_dump() 和 dict() 有什么区别？**
- A: 
  - `model_dump()`: Pydantic v2 方法，可配置参数多
  - `dict()`: Pydantic v1 方法，v2 中保留兼容
  - 推荐使用 `model_dump()`

**Q: 如何验证嵌套列表中的每个元素？**
- A: 使用 `field_validator` 配合 `mode='before'` 或 `mode='after'`：
  ```python
  @field_validator('emails', mode='after')
  def validate_emails(cls, v):
      for email in v:
          if '@' not in email:
              raise ValueError('invalid email')
      return v
  ```

**Q: Pydantic 如何处理循环引用？**
- A: 使用 `model_rebuild()` 延迟构建模型，如项目中的 `Message.model_rebuild()`。
  这是因为前向引用（forward references）需要特殊处理。

---

## 总结

本课程学习了 Pydantic 数据模型的核心知识：

| 知识点 | 说明 |
|--------|------|
| **BaseModel** | 所有模型的基类 |
| **Field** | 添加约束和元数据 |
| **验证器** | 自定义验证逻辑 |
| **嵌套模型** | 模型可以包含其他模型 |
| **JSON Schema** | 自动生成结构描述 |
| **序列化** | model_dump() / model_dump_json() |

**记住**：Pydantic 是 Agent 开发中处理数据的核心工具，特别是在 Function Calling 和 API 开发中！